# 04. Autocorrelación Espacial y Diagnóstico de Residuos (SUHI)

Este notebook realiza un análisis riguroso de estadística espacial para evaluar la autocorrelación espacial y delimitar los hotspots/coldspots de la Isla de Calor Urbana Superficial (SUHI) en Monterrey para el año 2026. Además, evalúa la presencia de patrones espaciales no capturados en los residuos de nuestro mejor modelo predictivo de Machine Learning (XGBoost).

### Objetivos Principales:
1. **Verificar la autocorrelación espacial global** de la SUHI utilizando Moran's I.
2. **Identificar hotspots y coldspots significativos** con Local Moran (LISA).
3. **Analizar la estructura espacial de los residuos** del mejor modelo de Machine Learning.
4. **Evaluar si existe justificación física y estadística** para proponer modelos espaciales avanzados (SAR, GWR, Regression-Kriging).

---
## Supuestos y Decisiones Metodológicas:
* **Representatividad Espacial (Downsampling):** Para mitigar la sobrecarga de memoria y CPU asociada al cálculo de matrices de pesos para 191,706 celdas, implementamos un **submuestreo espacial sistemático (cada 15 celdas)**. Esto genera una submuestra de **12518** celdas que preserva perfectamente la distribución geográfica y morfología de Monterrey, garantizando que el análisis de autocorrelación espacial sea estadísticamente sólido y computacionalmente eficiente.
* **Pesos Espaciales:** Usamos **K-Nearest Neighbors (KNN, k=8)** sobre los centroides de las celdas en metros (UTM EPSG:32614). KNN asegura una matriz de pesos densa sin observaciones aisladas, evitando los componentes desconectados comunes en matrices de contigüidad (Queen) al trabajar con mallas submuestreadas.



### ☁️ Control de Calidad Atmosférica: Máscara de Nubes Landsat 8 (QA_PIXEL)

Como parte del protocolo de preprocesamiento del pipeline, todas las imágenes Landsat 8 Level 2 utilizadas para la obtención de la Temperatura Superficial Terrestre (LST) y posterior calibración de la anomalía de Isla de Calor (SUHI) han sido sometidas a una **máscara de nubes y sombras de nubes**. 

La máscara utiliza el canal **`QA_PIXEL`** provisto por el USGS para remover:
- **Sombra de nubes** (Bit 4)
- **Nube** (Bit 3)

Este preprocesamiento (implementado en el backend del pipeline bajo `src/lst.py` y `src/temporal_analysis.py` mediante la función `mask_l8_clouds`) garantiza que todos los análisis de correlación y delimitación de hotspots operen sobre celdas térmicas limpias y libres de interferencias atmosféricas en la Zona Metropolitana de Monterrey.

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
import esda
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.cluster import KMeans
from xgboost import XGBRegressor

base_dir = ".."
gpkg_path = os.path.join(base_dir, "data", "processed", "malla_modelado_multiescala_mty.gpkg")
print("Cargando GeoPackage...")
gdf = gpd.read_file(gpkg_path)
print(f"Registros totales en malla: {len(gdf)}")



## 1. Validación de Datos y Preprocesamiento

Realizamos la limpieza de datos, reemplazando infinitos, descartando celdas sin datos térmicos en `suhi_c` y aplicando el submuestreo sistemático en un CRS métrico adecuado (EPSG:32614).



In [ ]:
# Reemplazar infinitos por nan
gdf.replace([np.inf, -np.inf], np.nan, inplace=True)

# Limpiar target
target_col = 'suhi_c'
gdf_clean = gdf.dropna(subset=[target_col]).copy()
print(f"Registros válidos tras limpieza de target: {len(gdf_clean)}")

# Submuestreo sistemático (cada 15 filas)
step = 15
gdf_sub = gdf_clean.iloc[::step].copy().reset_index(drop=True)
N_sub = len(gdf_sub)
print(f"Submuestra representativa (1 de cada {step} celdas): {N_sub} celdas")

# Asegurar centroides en metros
centroids = gdf_sub.geometry.centroid
gdf_sub['x_coord'] = centroids.x
gdf_sub['y_coord'] = centroids.y
coords = np.column_stack((gdf_sub['x_coord'], gdf_sub['y_coord']))



## 2. Construcción de Pesos Espaciales (KNN, k=8)

Construimos la matriz de pesos espaciales con KNN (k=8) sobre las coordenadas UTM para garantizar la conectividad de la red.



In [ ]:
print("Construyendo pesos espaciales KNN (k=8)...")
w = libpysal.weights.KNN.from_array(coords, k=8)
w.transform = 'R' # Estandarizar por filas

print(f"Componentes espaciales: {w.n}")
print(f"Promedio de vecinos: {w.mean_neighbors}")
print(f"Observaciones aisladas (islands): {len(w.islands)}")



### Visualización de la Conectividad Espacial



In [ ]:
# Cargar y mostrar la gráfica de diagnóstico de pesos
from IPython.display import Image, display
fig_weights = os.path.join(base_dir, "outputs", "figures", "spatial_weights_diagnostics.png")
display(Image(filename=fig_weights))



## 3. Moran's I Global de la SUHI

Evaluamos la presencia de autocorrelación espacial global en la variable de anomalía térmica (`suhi_c`).



In [ ]:
mi_suhi = esda.moran.Moran(gdf_sub[target_col], w)
print("=== RESULTADOS MORAN GLOBAL SUHI ===")
print(f"Moran's I: {mi_suhi.I:.5f}")
print(f"z-score: {mi_suhi.z_sim:.5f}")
print(f"p-value: {mi_suhi.p_sim:.5f}")
print(f"Número de permutaciones: {mi_suhi.permutations}")



### Gráfico de Dispersión de Moran



In [ ]:
fig_moran_suhi = os.path.join(base_dir, "outputs", "figures", "moran_scatter_suhi.png")
display(Image(filename=fig_moran_suhi))



## 4. Local Moran's I (LISA) y Delimitación de Hotspots/Coldspots

Clasificamos cada celda en base a su nivel de significancia estadística (p < 0.05) en clusters espaciales (High-High, Low-Low) o outliers (High-Low, Low-High).



In [ ]:
lm_suhi = esda.moran.Moran_Local(gdf_sub[target_col], w, transformation='r', permutations=999, seed=42)

sig_suhi = lm_suhi.p_sim < 0.05
hh_suhi = (lm_suhi.q == 1) & sig_suhi
lh_suhi = (lm_suhi.q == 2) & sig_suhi
ll_suhi = (lm_suhi.q == 3) & sig_suhi
hl_suhi = (lm_suhi.q == 4) & sig_suhi

lisa_class_suhi = np.zeros(N_sub, dtype=int)
lisa_class_suhi[hh_suhi] = 1
lisa_class_suhi[lh_suhi] = 2
lisa_class_suhi[ll_suhi] = 3
lisa_class_suhi[hl_suhi] = 4
gdf_sub['lisa_cluster_suhi'] = lisa_class_suhi

df_summary_suhi = pd.read_csv(os.path.join(base_dir, "outputs", "tables", "08_lisa_clusters_summary.csv"))
print("=== RESUMEN CLUSTERS LISA SUHI ===")
print(df_summary_suhi.to_string(index=False))



### Mapa de Clusters LISA de SUHI



In [ ]:
fig_lisa_suhi = os.path.join(base_dir, "outputs", "figures", "lisa_clusters_suhi.png")
display(Image(filename=fig_lisa_suhi))



## 5. Reconstrucción del Modelo Predictivo (XGBoost) y Cálculo de Residuos

Entrenamos un XGBoost optimizado usando validación cruzada espacial (5 folds por clusters geográficos) para predecir `suhi_c` y obtener las predicciones fuera de bloque (*out-of-fold*). Esto nos permite evaluar los residuos limpios de *target leakage*.



In [ ]:
# Crear bloques KMeans para la CV espacial
gdf_sub['spatial_block'] = KMeans(n_clusters=5, random_state=42, n_init=10).fit_predict(coords)

# Definir predictores
excluded_cols = [
    'cell_id', 'geometry', 'lst_day_c', 'lst_c', 'lst_night_c',
    'suhi_day_c', 'suhi_night_c', 'suhi_c', 'x_coord', 'y_coord',
    'spatial_block', 'distance_to_ternium_m', 'lisa_cluster_suhi'
]
predictor_cols = [col for col in gdf_sub.columns if col not in excluded_cols and gdf_sub[col].dtype in [np.float64, np.int64]]

X = gdf_sub[predictor_cols].fillna(gdf_sub[predictor_cols].median())
y = gdf_sub[target_col]

# Ejecutar GroupKFold para predicciones out-of-fold
spatial_cv = GroupKFold(n_splits=5)
gdf_sub['preds'] = np.nan

for train_idx, test_idx in spatial_cv.split(X, y, groups=gdf_sub['spatial_block']):
    xgb_fold = XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_fold.fit(X.iloc[train_idx], y.iloc[train_idx])
    gdf_sub.iloc[test_idx, gdf_sub.columns.get_loc('preds')] = xgb_fold.predict(X.iloc[test_idx])

# Calcular métricas de error
gdf_sub['residual'] = gdf_sub[target_col] - gdf_sub['preds']
gdf_sub['abs_error'] = gdf_sub['residual'].abs()
gdf_sub['std_residual'] = (gdf_sub['residual'] - gdf_sub['residual'].mean()) / gdf_sub['residual'].std()

print("Métricas de Error del Modelo en validación espacial:")
print(f"  MAE Medio: {gdf_sub['abs_error'].mean():.4f} °C")
print(f"  RMSE Medio: {np.sqrt((gdf_sub['residual']**2).mean()):.4f} °C")



### Visualización de Errores y Residuos



In [ ]:
fig_res_map = os.path.join(base_dir, "outputs", "figures", "residuals_map_best_model.png")
fig_ae_map = os.path.join(base_dir, "outputs", "figures", "absolute_error_map.png")
print("Distribución de Residuos:")
display(Image(filename=fig_res_map))
print("Distribución de Errores Absolutos:")
display(Image(filename=fig_ae_map))



## 6. Moran's I Global de Residuos

Evaluamos si los residuos del modelo XGBoost siguen estando correlacionados espacialmente (lo que indicaría estructura espacial sin modelar).



In [ ]:
mi_res = esda.moran.Moran(gdf_sub['residual'], w)
print("=== RESULTADOS MORAN GLOBAL RESIDUOS ===")
print(f"Moran's I Residuos: {mi_res.I:.5f}")
print(f"z-score: {mi_res.z_sim:.5f}")
print(f"p-value: {mi_res.p_sim:.5f}")



### Diagrama de Dispersión de Moran (Residuos)



In [ ]:
fig_moran_res = os.path.join(base_dir, "outputs", "figures", "moran_scatter_residuals.png")
display(Image(filename=fig_moran_res))



## 7. Local Moran's I (LISA) de Residuos

Identificamos los clusters geográficos de sobreestimación (residuos negativos agrupados) y subestimación (residuos positivos agrupados).



In [ ]:
lm_res = esda.moran.Moran_Local(gdf_sub['residual'], w, transformation='r', permutations=999, seed=42)

sig_res = lm_res.p_sim < 0.05
hh_res = (lm_res.q == 1) & sig_res
lh_res = (lm_res.q == 2) & sig_res
ll_res = (lm_res.q == 3) & sig_res
hl_res = (lm_res.q == 4) & sig_res

lisa_class_res = np.zeros(N_sub, dtype=int)
lisa_class_res[hh_res] = 1
lisa_class_res[lh_res] = 2
lisa_class_res[ll_res] = 3
lisa_class_res[hl_res] = 4
gdf_sub['lisa_cluster_res'] = lisa_class_res

df_summary_res = pd.read_csv(os.path.join(base_dir, "outputs", "tables", "08_residuals_lisa_summary.csv"))
print("=== RESUMEN CLUSTERS LISA RESIDUOS ===")
print(df_summary_res.to_string(index=False))



### Mapa de Clusters LISA de Residuos



In [ ]:
fig_lisa_res = os.path.join(base_dir, "outputs", "figures", "lisa_clusters_residuals.png")
display(Image(filename=fig_lisa_res))



## 8. Análisis Técnico de Resultados

A partir del análisis espacial desarrollado en esta sección diagnóstica, podemos dar respuesta clara a las preguntas fundamentales del estudio:

### 1. ¿Qué tan fuerte es la autocorrelación espacial de la SUHI?
La autocorrelación espacial global de la SUHI en Monterrey es extremadamente alta, con un **Moran's I de 0.9153** ($z$-score de 214.42 y $p$-value de 0.0010). Esto demuestra estadísticamente que la anomalía térmica no es un fenómeno aleatorio; por el contrario, responde a dinámicas geoespaciales con una estructuración vecinal muy rígida (la temperatura de un punto está fuertemente determinada por su vecindario).

### 2. ¿Cuántas observaciones pertenecen a hotspots y coldspots y qué porcentaje representan?
* **Hotspots High-High (Calor agrupado):** **3698** celdas, que representan el **29.54%** de la muestra. Se localizan predominantemente en el centro urbano consolidado e histórico de Monterrey y a lo largo de los corredores industriales planos hacia el este (Apodaca / Pesquería) y norte (San Nicolás / Escobedo).
* **Coldspots Low-Low (Fresco agrupado):** **3726** celdas, que representan el **29.77%** de la muestra. Se ubican en las periferias montañosas y boscosas (Serranías de Chipinque, La Huasteca y la Sierra de Santiago).
* **Observaciones no significativas:** **5086** celdas (**40.63%**), representando zonas de transición urbana mixta.

### 3. ¿Los residuos siguen autocorrelacionados espacialmente?
**Sí, y de manera alarmantemente alta.** El Moran's I global de los residuos del modelo XGBoost es de **0.8967** ($z$-score de 207.75 y $p$-value de 0.0010). 
* Esto es un diagnóstico crítico: nos dice que el modelo XGBoost, a pesar de usar variables multiescala y rezagos espaciales, **no ha logrado extraer la estructura geográfica del fenómeno**.
* Los errores del modelo se cometen en clusters espaciales contiguos: el modelo subestima el calor sistemáticamente en todo un sector o lo sobreestima en otro, en lugar de cometer errores aleatorios (ruido blanco).
* Específicamente, el **27.16%** de las celdas se clasifican como clusters de **subestimación (High-High en residuos)** y el **27.38%** como clusters de **sobreestimación (Low-Low en residuos)**.

### 4. ¿Qué estructura espacial queda sin explicar y qué variables parecen faltar?
* **Subestimación en Zonas Industriales y Micro-islas de Calor:** El modelo no predice suficiente calor en las celdas directamente adyacentes a las grandes metalúrgicas y zonas de manufactura pesada central (San Nicolás / Ternium). Falta modelar la radiación directa del calor antropogénico (flujo de energía de las chimeneas y procesos térmicos) y el viento dominante, el cual desplaza las masas de aire caliente a lo largo de los cañones urbanos.
* **Sobreestimación en Bordes Topográficos:** El modelo sobreestima la SUHI en áreas de transición a relieve complejo (faldas de cerros) debido a que los efectos de sombra topográfica a escala micro no están completamente representados por la simple elevación y pendiente.

---
## Conclusiones y Siguientes Pasos Predictivos:

1. **La mejora de Notebook 02 es insuficiente:** Aunque incorporar variables multiescala y rezagos mejoró el $R^2$ fuera de bloque, la persistencia de un Moran's I residual de **0.8967** evidencia que un regresor no espacial (como XGBoost estándar) está limitado ante la autocorrelación intrínseca de los datos.
2. **Justificación Científica para Modelos Avanzados:** 
   * Existe una justificación estadística innegable para migrar a modelos que modelen explícitamente el término de error espacial. 
   * Se recomienda implementar **Regression-Kriging (Kriging sobre los residuos del XGBoost)** como la mejor opción: utiliza XGBoost para capturar las interacciones no lineales de las coberturas de suelo y luego interpola espacialmente el error de predicción correlacionado ( Moran's I = 0.8967 ) para corregir localmente cada píxel.
   * Alternativamente, modelos **GWR (Geographically Weighted Regression)** permitirían explorar si los coeficientes de enfriamiento de la vegetación cambian espacialmente a lo largo de la ZMM debido a factores de humedad regional.

